# 12 · Comparación final de modelos

> **Viene de:** `07_tfidf_analysis`, `08_features_embeddings`, `09_deberta_finetuning`, `10_deberta_tuning` y `11_fusion_lightgbm_tuning` · **Lee:** `07_tfidf_grouped_results.csv`, `08_oof_predictions.csv`, `09_deberta_oof_predictions.csv`, `11_oof_predictions.csv` · **Produce:** `12_final_comparison.csv`

**Objetivo:** reunir en una sola tabla todos los modelos del proyecto, de mejor a peor, cada uno en su **versión
final**, es decir, después del ajuste de hiperparámetros cuando lo hubo.

Todas las cifras usan la misma validación: **Leave-One-Prompt-Out**, con un prompt que el modelo nunca vio. Por eso
son comparables entre sí.

## 1. Predicciones de cada notebook

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "src").is_dir() and (path / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("No se encontró la raíz del proyecto.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import regression_metrics

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
TARGETS: list[str] = ["content", "wording"]
PRED_COLUMNS: list[str] = [f"pred_{target}" for target in TARGETS]

# Todas las predicciones out-of-fold en formato largo: una fila por resumen y modelo
oof = pd.concat([pd.read_csv(PROCESSED_DATA_DIR / name) for name in
                 ["08_oof_predictions.csv", "09_deberta_oof_predictions.csv", "11_oof_predictions.csv"]],
                ignore_index=True)
reference = oof.loc[oof["modelo"] == "Dummy (media)", ["student_id", "prompt_id", *TARGETS]].reset_index(drop=True)
assert len(reference) == 7165


def predictions(model: str) -> np.ndarray:
    # Predicciones de un modelo en el mismo orden de resúmenes que `reference`.
    rows = reference[["student_id"]].merge(oof[oof["modelo"] == model], on="student_id", validate="one_to_one")
    return rows[PRED_COLUMNS].to_numpy()


y: np.ndarray = reference[TARGETS].to_numpy()
groups: np.ndarray = reference["prompt_id"].to_numpy()
print("Modelos con predicciones:", oof["modelo"].nunique(), "| resúmenes:", len(reference))

Modelos con predicciones: 14 | resúmenes: 7165


* 08, 09 y 11 guardaron sus predicciones out-of-fold con el mismo formato, así que se pueden juntar con `concat` y
  calcular todas las métricas con la misma función (`src.evaluation.regression_metrics`).
* `validate="one_to_one"` asegura que cada resumen tenga exactamente una predicción por modelo.
* Las predicciones de 07 no se guardaron en el repositorio, así que sus cifras se leen de su tabla de resultados, que
  usa la misma validación.

## 2. Tabla final

Cada modelo aparece **una sola vez, en su versión final**:

| Modelo | Versión que entra a la tabla |
|---|---|
| Fusión mínima + LightGBM | La ajustada con Optuna y validación anidada (11) |
| DeBERTa-v3-base | La elegida en el ajuste del 10: la selección por prompt eligió la configuración original del 09 |
| Promedio 50/50 | DeBERTa elegido + fusión mínima ajustada; el peso 50/50 se mantuvo porque ajustarlo no mejoró |
| BoW y TF-IDF | Los de 07, con `min_df` y `alpha` elegidos con búsqueda agrupada por prompt |
| Resto de 08 | Sin cambios: LightGBM con parámetros fijos y Ridge con `alpha` elegido por `RidgeCV` |

La columna `mcrmse_antes_del_ajuste` muestra el valor previo para los modelos que se ajustaron en 10 y 11.

In [2]:
DEBERTA: str = "DeBERTa-v3-base (fine-tuning)"
FUSION_08: str = "Opción 1: Fusión mínima (A + cos_sim) + LightGBM"
FUSION_11: str = "Fusión mínima + LightGBM ajustado (validación anidada)"
BLEND_11: str = "Promedio 50/50: DeBERTa + fusión ajustada"
FIXED_LGBM: str = "Parámetros fijos de antemano (sin ajuste)"
RIDGE_CV: str = "alpha elegido por RidgeCV en el entrenamiento de cada fold"
GROUPED_07: str = "min_df y alpha con búsqueda agrupada por prompt (anidada)"

# nombre en la tabla: (nombre en los archivos, notebook, cómo se ajustaron los hiperparámetros)
MODELS: dict[str, tuple[str, str, str]] = {
    "Promedio 50/50: DeBERTa + fusión mínima": (
        BLEND_11, "09–11", "Peso 50/50 fijo (ajustarlo no mejoró); componentes en su versión final"),
    "Fusión mínima (features + cos_sim) + LightGBM": (
        FUSION_11, "08 → 11", "Optuna (60 pruebas por búsqueda) con validación anidada por prompt"),
    "Fusión temprana (features + embedding) + LightGBM": (
        "Opción 1: Fusión temprana + LightGBM", "08", FIXED_LGBM),
    "Features manuales + LightGBM": ("A. Features + LightGBM", "08", FIXED_LGBM),
    "DeBERTa-v3-base (fine-tuning)": (
        DEBERTA, "09 → 10", "3 variantes (2 épocas, lr 1e-5, texto fuente); la selección por prompt eligió la original"),
    "Stacking (LightGBM features + Ridge embedding)": (
        "Opción 2: Stacking (LightGBM A + Ridge B)", "08", "Modelos base como en 08; meta-modelo Ridge(alpha=1)"),
    "Fusión temprana (features + embedding PCA) + Ridge": ("Opción 1: Fusión temprana + Ridge", "08", RIDGE_CV),
    "Features manuales + Ridge": ("A. Features + Ridge", "08", RIDGE_CV),
    "Solo largo + Ridge": ("Solo largo + Ridge", "08", RIDGE_CV),
    "Embedding sin tema + PCA(16) + Ridge": ("B. Embedding sin tema + PCA(16) + Ridge", "08", RIDGE_CV),
    "Embedding sin tema + Ridge": ("B. Embedding sin tema + Ridge", "08", RIDGE_CV),
    "Embedding crudo + Ridge": ("B. Embedding crudo + Ridge", "08", RIDGE_CV),
    "Dummy (media)": ("Dummy (media)", "08", "Sin hiperparámetros"),
}
# Valor antes del ajuste, para los modelos que se ajustaron en 10 y 11
BEFORE: dict[str, np.ndarray] = {
    "Promedio 50/50: DeBERTa + fusión mínima": 0.5 * predictions(DEBERTA) + 0.5 * predictions(FUSION_08),
    "Fusión mínima (features + cos_sim) + LightGBM": predictions(FUSION_08),
    "DeBERTa-v3-base (fine-tuning)": predictions(DEBERTA),
}


def spread_between_prompts(pred: np.ndarray) -> float:
    # Desviación estándar del MCRMSE de los 4 prompts: qué tanto cambia el error de un prompt a otro.
    return float(np.std([regression_metrics(y[groups == g], pred[groups == g])["mcrmse"]
                         for g in np.unique(groups)], ddof=1))


rows: list[dict] = []
for name, (source, notebook, tuning) in MODELS.items():
    pred = predictions(source)
    metrics = regression_metrics(y, pred)
    rows.append({"modelo": name, "notebook": notebook, "ajuste_hiperparametros": tuning,
                 "rmse_content": metrics["rmse_content"], "r2_content": metrics["r2_content"],
                 "rmse_wording": metrics["rmse_wording"], "r2_wording": metrics["r2_wording"],
                 "mcrmse": metrics["mcrmse"],
                 "mcrmse_antes_del_ajuste": regression_metrics(y, BEFORE[name])["mcrmse"] if name in BEFORE else np.nan,
                 "desviacion_entre_prompts": spread_between_prompts(pred)})

lexical = pd.read_csv(TABLES_DIR / "07_tfidf_grouped_results.csv")
lexical = lexical[lexical["modelo"] != "Dummy media"]   # el Dummy ya está, calculado en 08
for record in lexical.to_dict("records"):
    rows.append({"modelo": record["modelo"], "notebook": "07", "ajuste_hiperparametros": GROUPED_07,
                 **{key: record[key] for key in ["rmse_content", "r2_content", "rmse_wording", "r2_wording", "mcrmse"]},
                 "mcrmse_antes_del_ajuste": np.nan,
                 "desviacion_entre_prompts": record["desviacion_mcrmse_entre_prompts"]})

final = pd.DataFrame(rows).sort_values("mcrmse").reset_index(drop=True)
dummy = final.loc[final["modelo"] == "Dummy (media)", "mcrmse"].item()
final.insert(0, "posicion", np.arange(1, len(final) + 1))
final["mejora_vs_dummy_%"] = 100 * (1 - final["mcrmse"] / dummy)
final.to_csv(TABLES_DIR / "12_final_comparison.csv", index=False)

with pd.option_context("display.max_colwidth", 90, "display.width", 250):
    display(final.set_index("posicion").round(3))

,modelo,notebook,ajuste_hiperparametros,rmse_content,r2_content,rmse_wording,r2_wording,mcrmse,mcrmse_antes_del_ajuste,desviacion_entre_prompts,mejora_vs_dummy_%
posicion,,,,,,,,,,,
1,Promedio 50/50: DeBERTa + fusión mínima,09–11,Peso 50/50 fijo (ajustarlo no mejoró); componentes en su versión final,0.433,0.828,0.626,0.635,0.529,0.529,0.048,49.602
2,Fusión mínima (features + cos_sim) + LightGBM,08 → 11,Optuna (60 pruebas por búsqueda) con validación anidada por prompt,0.478,0.790,0.669,0.583,0.573,0.576,0.081,45.390
3,Fusión temprana (features + embedding) + LightGBM,08,Parámetros fijos de antemano (sin ajuste),0.487,0.782,0.668,0.584,0.578,NaN,0.094,44.991
4,Features manuales + LightGBM,08,Parámetros fijos de antemano (sin ajuste),0.485,0.784,0.677,0.573,0.581,NaN,0.085,44.644
5,DeBERTa-v3-base (fine-tuning),09 → 10,"3 variantes (2 épocas, lr 1e-5, texto fuente); la selección por prompt eligió la original",0.462,0.804,0.740,0.490,0.601,0.601,0.057,42.791
6,Stacking (LightGBM features + Ridge embedding),08,Modelos base como en 08; meta-modelo Ridge(alpha=1),0.505,0.765,0.701,0.542,0.603,NaN,0.080,42.545
7,Fusión temprana (features + embedding PCA) + Ridge,08,alpha elegido por RidgeCV en el entrenamiento de cada fold,0.504,0.767,0.710,0.530,0.607,NaN,0.076,42.194
8,Features manuales + Ridge,08,alpha elegido por RidgeCV en el entrenamiento de cada fold,0.510,0.761,0.713,0.526,0.611,NaN,0.082,41.762
9,Solo largo + Ridge,08,alpha elegido por RidgeCV en el entrenamiento de cada fold,0.561,0.711,0.862,0.307,0.712,NaN,0.087,32.227


* `MODELS` relaciona el nombre que aparece en la tabla con el nombre que tiene el modelo en los archivos de
  predicciones. Así la tabla final usa nombres más claros sin cambiar los archivos de los notebooks anteriores.
* `ddof=1` calcula la desviación estándar muestral, igual que la columna equivalente de 08.
* `mejora_vs_dummy_%` es cuánto baja el MCRMSE respecto a predecir siempre la media (100 % sería error cero).
* En el promedio 50/50, la versión final usa la fusión ajustada (0.5291) y la de antes la fusión de 08 (0.5285): la
  diferencia es menor a 0.001 y no cambia ninguna conclusión.

## Puente

Esta tabla es la que va al informe. El modelo recomendado es el primero, el promedio 50/50 de DeBERTa y la fusión
mínima. El mejor modelo individual es la fusión mínima con LightGBM, que además es mucho más barato: segundos en CPU
contra unos 33 minutos de GPU para DeBERTa.